# Atividade: Integração do Teachable Machine com Aplicação Simples

Nesta atividade, o objetivo foi utilizar o **Teachable Machine** do Google ([https://teachablemachine.withgoogle.com/](https://teachablemachine.withgoogle.com/)) para criar um modelo treinado com base em reconhecimento de **imagem** e integrá-lo a uma aplicação simples, que utiliza o resultado da predição para tomar uma decisão no fluxo do programa.

---

## Descrição da Atividade

Utilizei o Teachable Machine para treinar um modelo capaz de **reconhecer os escudos dos 20 clubes da Série A do Campeonato Brasileiro**. Cada classe representa um dos clubes participantes da temporada atual.

O fluxo da aplicação funciona da seguinte forma:

1. O usuário fornece uma imagem com o escudo de um dos clubes.
2. A aplicação utiliza o modelo treinado para identificar qual clube está representado na imagem.
3. Com base no clube identificado, o programa retorna a **posição atual do clube na tabela do Campeonato Brasileiro** (posições definidas manualmente conforme o momento da atividade).

Essa integração entre o Teachable Machine e uma aplicação externa demonstra como modelos treinados podem ser aplicados em contextos práticos e interativos, mesmo em projetos simples.

---

## Link para o vídeo demonstrativo

📹 [https://youtu.be/N4NqjsTRg_Q](https://youtu.be/N4NqjsTRg_Q)

---

*Observação:* O modelo foi exportado do Teachable Machine como TensorFlow (formato Keras) e carregado no Colab para fazer as predições a partir de imagens fornecidas pelo usuário.


In [ ]:
# imports
from google.colab import drive, files
from PIL import Image
import tensorflow as tf
import numpy as np
import os

In [ ]:
# @title
# mocks

data = {
    "standings": [
        {
            "tournament": {
                "name": "Brasileirão Betano",
                "slug": "brasileirao-serie-a",
                "category": {
                    "id": 13,
                    "country": {
                        "alpha2": "BR",
                        "alpha3": "BRA",
                        "name": "Brazil",
                        "slug": "brazil"
                    },
                    "name": "Brazil",
                    "slug": "brazil",
                    "sport": {
                        "name": "Football",
                        "slug": "football",
                        "id": 1
                    },
                    "flag": "brazil",
                    "alpha2": "BR"
                },
                "uniqueTournament": {
                    "name": "Brasileirão Betano",
                    "slug": "brasileirao-serie-a",
                    "primaryColorHex": "#C7FF00",
                    "secondaryColorHex": "#969696",
                    "category": {
                        "id": 13,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "name": "Brazil",
                        "slug": "brazil",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "flag": "brazil",
                        "alpha2": "BR"
                    },
                    "userCount": 315464,
                    "hasPerformanceGraphFeature": True,
                    "id": 325,
                    "country": {},
                    "displayInverseHomeAwayTeams": False
                },
                "priority": 503,
                "isGroup": False,
                "isLive": False,
                "id": 83
            },
            "name": "Brasileiro Serie A 2025",
            "type": "total",
            "descriptions": [],
            "tieBreakingRule": {
                "text": "In the event that two (or more) teams have an equal number of points, the following rules break the tie:\n\n1. Number of victories\n2. Goal difference\n3. Goals scored",
                "id": 1347
            },
            "rows": [
                {
                    "team": {
                        "name": "Palmeiras",
                        "slug": "palmeiras",
                        "shortName": "Palmeiras",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 547911,
                        "nameCode": "PAL",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1963,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#339966",
                            "secondary": "#336633",
                            "text": "#336633"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Палмейрас"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Libertadores",
                        "id": 19
                    },
                    "position": 1,
                    "matches": 8,
                    "wins": 6,
                    "scoresFor": 9,
                    "scoresAgainst": 3,
                    "id": 1436830,
                    "losses": 1,
                    "draws": 1,
                    "points": 19,
                    "scoreDiffFormatted": "+6"
                },
                {
                    "team": {
                        "name": "Flamengo",
                        "slug": "flamengo",
                        "shortName": "Flamengo",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 810221,
                        "nameCode": "FLA",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 5981,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#ff0000",
                            "secondary": "#000000",
                            "text": "#000000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "ФК Фламенго РЖ"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Libertadores",
                        "id": 19
                    },
                    "position": 2,
                    "matches": 8,
                    "wins": 5,
                    "scoresFor": 17,
                    "scoresAgainst": 4,
                    "id": 1436814,
                    "losses": 1,
                    "draws": 2,
                    "points": 17,
                    "scoreDiffFormatted": "+13"
                },
                {
                    "team": {
                        "name": "Red Bull Bragantino",
                        "slug": "red-bull-bragantino",
                        "shortName": "RB Bragantino",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 145838,
                        "nameCode": "BRA",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1999,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#363636",
                            "secondary": "#d8d8d6",
                            "text": "#d8d8d6"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Ред Булл Брагантино"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Libertadores",
                        "id": 19
                    },
                    "position": 3,
                    "matches": 8,
                    "wins": 5,
                    "scoresFor": 10,
                    "scoresAgainst": 6,
                    "id": 1436824,
                    "losses": 1,
                    "draws": 2,
                    "points": 17,
                    "scoreDiffFormatted": "+4"
                },
                {
                    "team": {
                        "name": "Cruzeiro",
                        "slug": "cruzeiro",
                        "shortName": "Cruzeiro",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 262675,
                        "nameCode": "CRU",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1954,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#0033cc",
                            "secondary": "#0033cc",
                            "text": "#0033cc"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Крузейро"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Libertadores",
                        "id": 19
                    },
                    "position": 4,
                    "matches": 8,
                    "wins": 5,
                    "scoresFor": 13,
                    "scoresAgainst": 7,
                    "id": 1436816,
                    "losses": 2,
                    "draws": 1,
                    "points": 16,
                    "scoreDiffFormatted": "+6"
                },
                {
                    "team": {
                        "name": "Fluminense",
                        "slug": "fluminense",
                        "shortName": "Fluminense",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 310909,
                        "nameCode": "FLU",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1961,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#660000",
                            "secondary": "#006633",
                            "text": "#006633"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Флуминенсе"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Libertadores Qualification",
                        "id": 20
                    },
                    "position": 5,
                    "matches": 8,
                    "wins": 4,
                    "scoresFor": 10,
                    "scoresAgainst": 10,
                    "id": 1436820,
                    "losses": 3,
                    "draws": 1,
                    "points": 13,
                    "scoreDiffFormatted": "0"
                },
                {
                    "team": {
                        "name": "Ceará",
                        "slug": "ceara",
                        "shortName": "Ceará",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 89206,
                        "nameCode": "CEA",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 2001,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#000000",
                            "secondary": "#ffffff",
                            "text": "#ffffff"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "ФК Сеара"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Libertadores Qualification",
                        "id": 20
                    },
                    "position": 6,
                    "matches": 8,
                    "wins": 3,
                    "scoresFor": 9,
                    "scoresAgainst": 7,
                    "id": 1436813,
                    "losses": 2,
                    "draws": 3,
                    "points": 12,
                    "scoreDiffFormatted": "+2"
                },
                {
                    "team": {
                        "name": "Atlético Mineiro",
                        "slug": "atletico-mineiro",
                        "shortName": "Atlético-MG",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 295400,
                        "nameCode": "ATL",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1977,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#000000",
                            "secondary": "#ffffff",
                            "text": "#ffffff"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Атлетико Минейро"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Sudamericana",
                        "id": 21
                    },
                    "position": 7,
                    "matches": 8,
                    "wins": 3,
                    "scoresFor": 10,
                    "scoresAgainst": 10,
                    "id": 1436811,
                    "losses": 2,
                    "draws": 3,
                    "points": 12,
                    "scoreDiffFormatted": "0"
                },
                {
                    "team": {
                        "name": "Bahia",
                        "slug": "bahia",
                        "shortName": "Bahia",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 229594,
                        "nameCode": "BAH",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1955,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#ffffff",
                            "secondary": "#333399",
                            "text": "#333399"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Баия"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Sudamericana",
                        "id": 21
                    },
                    "position": 8,
                    "matches": 8,
                    "wins": 3,
                    "scoresFor": 7,
                    "scoresAgainst": 8,
                    "id": 1436817,
                    "losses": 2,
                    "draws": 3,
                    "points": 12,
                    "scoreDiffFormatted": "-1"
                },
                {
                    "team": {
                        "name": "Botafogo",
                        "slug": "botafogo",
                        "shortName": "Botafogo",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 311580,
                        "nameCode": "BOT",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1958,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#000000",
                            "secondary": "#ffffff",
                            "text": "#ffffff"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "ФК Ботафого"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Sudamericana",
                        "id": 21
                    },
                    "position": 9,
                    "matches": 8,
                    "wins": 3,
                    "scoresFor": 10,
                    "scoresAgainst": 5,
                    "id": 1436812,
                    "losses": 3,
                    "draws": 2,
                    "points": 11,
                    "scoreDiffFormatted": "+5"
                },
                {
                    "team": {
                        "name": "Corinthians",
                        "slug": "corinthians",
                        "shortName": "Corinthians",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 580014,
                        "nameCode": "COR",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1957,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#ffffff",
                            "secondary": "#000000",
                            "text": "#000000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Коринтианс"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Sudamericana",
                        "id": 21
                    },
                    "position": 10,
                    "matches": 8,
                    "wins": 3,
                    "scoresFor": 11,
                    "scoresAgainst": 14,
                    "id": 1436827,
                    "losses": 4,
                    "draws": 1,
                    "points": 10,
                    "scoreDiffFormatted": "-3"
                },
                {
                    "team": {
                        "name": "Fortaleza",
                        "slug": "fortaleza",
                        "shortName": "Fortaleza",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 198202,
                        "nameCode": "FOR",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 2020,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#0000ff",
                            "secondary": "#ff0000",
                            "text": "#ff0000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Форталеза"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Sudamericana",
                        "id": 21
                    },
                    "position": 11,
                    "matches": 8,
                    "wins": 2,
                    "scoresFor": 10,
                    "scoresAgainst": 5,
                    "id": 1436821,
                    "losses": 2,
                    "draws": 4,
                    "points": 10,
                    "scoreDiffFormatted": "+5"
                },
                {
                    "team": {
                        "name": "Mirassol",
                        "slug": "mirassol",
                        "shortName": "Mirassol",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 33669,
                        "nameCode": "MIR",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 21982,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#e8c728",
                            "secondary": "#174c30",
                            "text": "#174c30"
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Copa Sudamericana",
                        "id": 21
                    },
                    "position": 12,
                    "matches": 8,
                    "wins": 2,
                    "scoresFor": 13,
                    "scoresAgainst": 11,
                    "id": 1436823,
                    "losses": 2,
                    "draws": 4,
                    "points": 10,
                    "scoreDiffFormatted": "+2"
                },
                {
                    "team": {
                        "name": "Internacional",
                        "slug": "internacional",
                        "shortName": "Internacional",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 238984,
                        "nameCode": "INT",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1966,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#cc0000",
                            "secondary": "#cc0000",
                            "text": "#cc0000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Интернаcьонал"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "position": 13,
                    "matches": 8,
                    "wins": 2,
                    "scoresFor": 10,
                    "scoresAgainst": 12,
                    "id": 1436828,
                    "losses": 3,
                    "draws": 3,
                    "points": 9,
                    "scoreDiffFormatted": "-2"
                },
                {
                    "team": {
                        "name": "Vitória",
                        "slug": "vitoria",
                        "shortName": "Vitória",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 126076,
                        "nameCode": "VIT",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1962,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#ff0000",
                            "secondary": "#000000",
                            "text": "#000000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Виториа БА"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "position": 14,
                    "matches": 8,
                    "wins": 2,
                    "scoresFor": 9,
                    "scoresAgainst": 11,
                    "id": 1436819,
                    "losses": 3,
                    "draws": 3,
                    "points": 9,
                    "scoreDiffFormatted": "-2"
                },
                {
                    "team": {
                        "name": "Grêmio",
                        "slug": "gremio",
                        "shortName": "Grêmio",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 270844,
                        "nameCode": "GPA",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 5926,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#3399ff",
                            "secondary": "#000033",
                            "text": "#000033"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Гремио"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "position": 15,
                    "matches": 8,
                    "wins": 2,
                    "scoresFor": 7,
                    "scoresAgainst": 12,
                    "id": 1436822,
                    "losses": 3,
                    "draws": 3,
                    "points": 9,
                    "scoreDiffFormatted": "-5"
                },
                {
                    "team": {
                        "name": "São Paulo",
                        "slug": "sao-paulo",
                        "shortName": "São Paulo",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 456663,
                        "nameCode": "SPA",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1981,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#ffffff",
                            "secondary": "#000000",
                            "text": "#000000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "ФК Сан-Паулу"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "position": 16,
                    "matches": 8,
                    "wins": 1,
                    "scoresFor": 6,
                    "scoresAgainst": 6,
                    "id": 1436826,
                    "losses": 1,
                    "draws": 6,
                    "points": 9,
                    "scoreDiffFormatted": "0"
                },
                {
                    "team": {
                        "name": "Vasco da Gama",
                        "slug": "vasco-da-gama",
                        "shortName": "Vasco",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 331772,
                        "nameCode": "VAS",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1974,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#ffffff",
                            "secondary": "#000000",
                            "text": "#000000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Васко да Гама"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Relegation",
                        "id": 3
                    },
                    "position": 17,
                    "matches": 8,
                    "wins": 2,
                    "scoresFor": 7,
                    "scoresAgainst": 11,
                    "id": 1436815,
                    "losses": 5,
                    "draws": 1,
                    "points": 7,
                    "scoreDiffFormatted": "-4"
                },
                {
                    "team": {
                        "name": "Juventude",
                        "slug": "juventude",
                        "shortName": "Juventude",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 73358,
                        "nameCode": "JUV",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1980,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#006600",
                            "secondary": "#ffffff",
                            "text": "#ffffff"
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Relegation",
                        "id": 3
                    },
                    "position": 18,
                    "matches": 8,
                    "wins": 2,
                    "scoresFor": 7,
                    "scoresAgainst": 20,
                    "id": 1436818,
                    "losses": 5,
                    "draws": 1,
                    "points": 7,
                    "scoreDiffFormatted": "-13"
                },
                {
                    "team": {
                        "name": "Santos",
                        "slug": "santos",
                        "shortName": "Santos",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 390181,
                        "nameCode": "SAN",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1968,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#ffffff",
                            "secondary": "#ffffff",
                            "text": "#ffffff"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "ФК Сантос"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Relegation",
                        "id": 3
                    },
                    "position": 19,
                    "matches": 8,
                    "wins": 1,
                    "scoresFor": 7,
                    "scoresAgainst": 10,
                    "id": 1436825,
                    "losses": 5,
                    "draws": 2,
                    "points": 5,
                    "scoreDiffFormatted": "-3"
                },
                {
                    "team": {
                        "name": "Sport Recife",
                        "slug": "sport-recife",
                        "shortName": "Sport",
                        "gender": "M",
                        "sport": {
                            "name": "Football",
                            "slug": "football",
                            "id": 1
                        },
                        "userCount": 112111,
                        "nameCode": "SPO",
                        "disabled": False,
                        "national": False,
                        "type": 0,
                        "id": 1959,
                        "country": {
                            "alpha2": "BR",
                            "alpha3": "BRA",
                            "name": "Brazil",
                            "slug": "brazil"
                        },
                        "teamColors": {
                            "primary": "#cc0000",
                            "secondary": "#000000",
                            "text": "#000000"
                        },
                        "fieldTranslations": {
                            "nameTranslation": {
                                "ru": "Спорт Ресифи"
                            },
                            "shortNameTranslation": {}
                        }
                    },
                    "descriptions": [],
                    "promotion": {
                        "text": "Relegation",
                        "id": 3
                    },
                    "position": 20,
                    "matches": 8,
                    "wins": 0,
                    "scoresFor": 4,
                    "scoresAgainst": 14,
                    "id": 1436829,
                    "losses": 6,
                    "draws": 2,
                    "points": 2,
                    "scoreDiffFormatted": "-10"
                }
            ],
            "id": 157134,
            "updatedAtTimestamp": 1739454210
        }
    ]
}

In [ ]:
# get file from drive
def get_model_and_labels_from_drive():

  drive.mount('/content/drive')

  tflite_model_path = '/content/drive/MyDrive/flag-detector.tflite'
  label_path = '/content/drive/MyDrive/flag-detector-labels.txt'

  with open(label_path, 'r') as f:
    labels = f.read().split('\n')

  interpreter = tf.lite.Interpreter(model_path=tflite_model_path)
  interpreter.allocate_tensors()

  return labels, interpreter

# convert an image to an array
def preprocess_image(input_shape):

  image_uploaded = files.upload()
  image_path = list(image_uploaded.keys())[0]

  image = Image.open(image_path).convert('RGB')
  image = image.resize((input_shape[1], input_shape[2]))

  image_array = np.array(image).astype(np.float32) / 255.0
  image_array = np.expand_dims(image_array, axis=0)

  os.remove(image_path)

  return image_array

# make a prediction using the image array
def predict_image(interpreter, labels, image_array, threshold=0.5):

  interpreter.set_tensor(interpreter.get_input_details()[0]['index'], image_array)
  interpreter.invoke()
  output_data = interpreter.get_tensor(interpreter.get_output_details()[0]['index'])

  prediction = np.squeeze(output_data)
  max_index = np.argmax(prediction)
  confidence = prediction[max_index]

  if confidence < threshold:
      return "Não reconhecido"
  else:
      return labels[max_index], confidence

# mock the sofascore api
def sofascore_api(team_name):
  # url = 'https://www.sofascore.com/api/v1/unique-tournament/325/season/72034/standings/total'

  if team_name == 'Não reconhecido':
    return 'Não foi possível obter nenhuma informação para um time não reconhecido.'

  table = data.get("standings")[0]

  for row in table.get("rows"):
      if row["team"]["name"].lower() == team_name.lower():
          return {
              "Time": row["team"]["name"],
              "Posição": row["position"],
              "Pontos": row["points"],
              "Vitórias": row["wins"],
              "Empates": row["draws"],
              "Derrotas": row["losses"],
              "Gols pró": row["scoresFor"],
              "Gols contra": row["scoresAgainst"],
              "Saldo de gols": row["scoreDiffFormatted"]
          }

  return f"Time '{team_name}' não encontrado na tabela."

In [ ]:
labels, interpreter = get_model_and_labels_from_drive()
image_array = preprocess_image(interpreter.get_input_details()[0]['shape'])
team_name, confidence = predict_image(interpreter, labels, image_array)

print(f"Classe: {team_name} (Confiança: {confidence:.2%})")
print(sofascore_api(team_name))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Saving sport.png to sport.png
Classe: Sport Recife (Confiança: 97.90%)
{'Time': 'Sport Recife', 'Posição': 20, 'Pontos': 2, 'Vitórias': 0, 'Empates': 2, 'Derrotas': 6, 'Gols pró': 4, 'Gols contra': 14, 'Saldo de gols': '-10'}
